# Structure-Based Bioactivity Grading - Third Phase

Small-scale docking validation of the ML-QSAR selected compounds.

Runs locally on a personal computer (<20min runtime, 80 compounds).

**Inputs:**
- `Imidazolones_30samples.csv` (ID, SMILES, QSAR_score)
- `Thiazolones_15samples.csv` (ID, SMILES, QSAR_score)
- `6COX.pdb` (COX-2, ligand S58)
- `3KK6.pdb` (COX-1, ligand FLC)

**Workflow:**
1. **PREPARE** — Ligand (multi-conformer) + receptor preparation
2. **DOCK** — AutoDock Vina (exhaustiveness=16, num_modes=3)
3. **SCORE** — Geometric scoring, pose selection, final ranking

***(Please read [the third phase README.md](03_docking_pdbqts_data/README.md) for full details on this notebook).***


In [ ]:
import sys
import pandas as pd

sys.path.insert(0, "03_docking_pdbqts_data")
from modules import docking as dock

cfg = dock.get_docking_config()
dirs = dock.init_hpc_dirs(
    cfg["ROOT"] / "03_docking_pdbqts_data/.interim",
    receptor_ids=["6COX", "3KK6"],
)

## 📥 17. Load input compounds

Load the QSAR-prioritized compounds from file.
Thiazolones and imidazolones are loaded separately then concatenated.


In [ ]:
df_ligands_raw = dock.load_ligands([
    cfg["INPUT_CSV_THIAZOLONES"],
    cfg["INPUT_CSV_IMIDAZOLONES"],
])
display(df_ligands_raw[["ID", "SMILES", "QSAR_score"]].head())

## ⬥ 18. Prepare ligands (multi-conformer)

Generate RDKit multi-conformer structures for flexible docking.


In [ ]:
df_ligands = dock.prepare_ligands_multi_conf(
    df_ligands_raw, dirs["ligands"], n_confs=15, seed=42,
)

## ⬥ 19. Prepare receptors

Prepare COX-2 (6COX) and COX-1 (3KK6) with binding site boxes.


In [ ]:
cox2_info, cox1_info = dock.prepare_all_receptors(dirs, cfg)

## 🔹 20. Run docking

Create ligand–receptor mapping and run AutoDock Vina locally.


In [ ]:
mapping_df, docking_stats = dock.run_docking_workflow(
    df_ligands, dirs, exhaustiveness=16, num_modes=3, seed=42,
)

## ⬥ 21. Validate and parse results

Check docking completion and extract all ligand poses.


In [ ]:
df_all_poses = dock.validate_and_extract(
    dirs, ["6COX", "3KK6"], cfg["RECEPTOR_MAP"], num_modes=3,
)
if not df_all_poses.empty:
    display(df_all_poses.head(10))

## 🔸 22. Geometric scoring and pose selection

Select the best pose per compound using geometric criteria.


In [ ]:
df_best_poses = dock.select_best_poses_across(
    df_all_poses, dirs, ["6COX", "3KK6"], cfg["receptor_pdb_map"],
)
if not df_best_poses.empty:
    display(df_best_poses.head(10))

## ⬥ 24. Final ranking

Combine QSAR scores with docking scores into a unified ranking.


In [ ]:
df_ranked = dock.compute_final_ranking(df_best_poses, df_ligands_raw)
if not df_ranked.empty:
    cols = ["ligand_id", "QSAR_score", "score_cox2", "score_cox1",
            "geometric_score", "final_score"]
    display(df_ranked[[c for c in cols if c in df_ranked.columns]].round(4))

## 📤 25. Output

Save the ranked scores and poses.


In [ ]:
dock.save_all_outputs(
    df_ranked, dirs,
    cfg["ROOT"] / "03_docking_pdbqts_data/outputs",
)

## 🖼️ 26. Visualization (optional)

Render top-ranked poses with PyMOL.


In [ ]:
dock.render_top_poses(
    df_ranked, dirs, receptor_id="6COX",
    receptor_pdb=cfg["COX2_PDB"],
    output_dir=cfg["ROOT"] / "03_docking_pdbqts_data/outputs/figures",
    top_n=5,
)